In [5]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import warnings 
warnings.filterwarnings("ignore")

In [8]:
model_name = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!")

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 2827.71it/s]


Model loaded successfully!


In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [ ]:
data = [
    {"instruction": "Fine-tuning kya hoti hai?", "output": "Fine-tuning ek technique hai jisme pre-trained model ko naye specific data par train karke kisi task ke liye customize kiya jata hai."},
    {"instruction": "Python kya hai?", "output": "Python ek high-level programming language hai jo simplicity aur readability ke liye jaani jaati hai."},
    {"instruction": "AI kya hota hai?", "output": "AI ka matlab Artificial Intelligence hai, jisme machines insano jaisi intelligence dikhati hain."},
    {"instruction": "Machine Learning kya hai?", "output": "Machine Learning AI ka ek subset hai jisme model data se patterns seekhta hai bina explicitly program kiye."},
    {"instruction": "LoRA kya hai?", "output": "LoRA ek fine-tuning technique hai jisme model ke saare parameters update karne ke bajaye sirf chhote extra parameters train hote hain."},
    # Yaha aap apne 50-200+ examples add karo — jitna zyada quality data, utna   better result
]

with open("data.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"{len(data)} examples saved to data.jsonl")

5 examples saved to data.jsonl


In [11]:
dataset = load_dataset("json", data_files="data.jsonl", split="train")

def format_example(example):
    example["text"] = f"### Instruction:\n{example['instruction']}\n### Output:\n{example['output']}{tokenizer.eos_token}"
    return example

dataset = dataset.map(format_example)
print(dataset[0]["text"])

Generating train split: 5 examples [00:00, 249.63 examples/s]
Map: 100%|██████████| 5/5 [00:00<00:00, 114.69 examples/s]

### Instruction:
Fine-tuning kya hoti hai?
### Output:
Fine-tuning ek technique hai jisme pre-trained model ko naye specific data par train karke kisi task ke liye customize kiya jata hai.<|end_of_text|>


In [13]:
config = SFTConfig(
    output_dir="./output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    max_length=256,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    use_cpu=True,
    bf16=False,
    fp16=False
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=config
)

Truncating train dataset: 100%|██████████| 5/5 [00:00<00:00, 115.74 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 5/5 [00:00<00:00, 157.53 examples/s]


In [14]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
1,3.862265
2,3.830091
3,4.743578
4,3.529987
5,3.547252
6,4.733536
7,3.534501
8,3.818123
9,3.837949
10,3.516332


TrainOutput(global_step=15, training_loss=3.890223582585653, metrics={'train_runtime': 20689.0499, 'train_samples_per_second': 0.001, 'train_steps_per_second': 0.001, 'total_flos': 3594052915200.0, 'train_loss': 3.890223582585653, 'epoch': 3.0})

In [15]:
model.save_pretrained("./fine-tuned-llama")
tokenizer.save_pretrained("./fine-tuned-llama")
print("Saved!")

Saved!


In [16]:
from peft import PeftModel

# Test input
prompt = "### Instruction:\nFine-tuning kya hoti hai?\n### Output:\n"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### Instruction:
Fine-tuning kya hoti hai?
### Output:
K--Title and the
def,def a def
def.def and def,Tags and the the a that and of.
from for the,##.# for.def the of and you and) of the: the.def to.def in Question and the# in the the or for
Question in and and the def from.defdef,def.def to
def#.def def.from to the.#,def)
